In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [3]:
## Chatopenai
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
response = model.invoke("why do parrots talk?")
response

AIMessage(content=[{'type': 'text', 'text': 'Parrots talk primarily because they are **highly social creatures that rely on mimicry to survive and bond with their flock.**\n\nWhile humans speak to exchange complex information, parrots use their vocal mimicry to navigate their environment. Here is a breakdown of why they do it:\n\n### 1. Social Bonding (The "Flock" Instinct)\nIn the wild, parrots live in tight-knit social groups. To keep the flock together, they use contact calls to identify each other. They often learn the specific vocalizations of their mates or family members to strengthen their bonds. When a parrot lives with a human, the human becomes its "flock." The parrot mimics the human’s sounds because it wants to fit in and communicate with its new family.\n\n### 2. Survival and Integration\nMimicry is an evolutionary tool. In the wild, parrots are surrounded by various sounds—other birds, predators, and environmental noises. By being able to replicate the sounds of their en

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""   #Docstring which explain the function
    return f"It's sunny in {location}"

model_with_tool = model.bind_tools([get_weather])

In [5]:
response = model_with_tool.invoke("What's the weather like in boston?")
print(response)

for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "boston"}'}, '__gemini_function_call_thought_signatures__': {'25dtUonQ': 'EjQKMgERTTIP1FrEdtHmtA6NpmptFbxSPPby5R4S13UrWuQfxk0/q0fVZONtYMuWbFDmTqt5'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f514d-377c-7e01-b507-7b91bcf6c495-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': '25dtUonQ', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Args: {'location': 'boston'}


### Tool Execution Loops


In [6]:
# step1: Model generates tool calls
messages = [{"role":"user", "content": "What's the weather in Boston"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

#step2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_results = get_weather.invoke(tool_call)
    messages.append(tool_results)
    
#step3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)

The weather in Boston is currently sunny.
